# SMB-lending diligence: runway and ranking

This notebook recomputes cash runway for every company in `analysis/comparison.json` and ranks the cohort for the Q3 2026 credit committee.

**Runway (months) = cash and equivalents / average monthly net burn**, both in USD millions and both cited in the table.

In [ ]:
import json
from pathlib import Path

table = json.loads(Path("comparison.json").read_text())
rows = table["rows"]
print(len(rows), "companies as of", table["asOf"])

## Recompute runway

The stored `runwayMonths` must equal cash / burn within rounding (0.1 month). Any mismatch means the table was edited by hand.

In [ ]:
for r in rows:
    runway = r["cashUsdM"]["value"] / r["monthlyBurnUsdM"]["value"]
    assert abs(runway - r["runwayMonths"]["value"]) <= 0.1, r["company"]
    r["runway_check"] = round(runway, 1)
[(r["company"], r["runway_check"]) for r in rows]

## Ranking

Candidates are ranked by runway, longest first (committee criteria). Runway below 12 months needs an equity commitment or a liquidity covenant; below 9 months is a decline unless new equity is committed.

In [ ]:
def band(runway):
    return "proceed" if runway >= 12 else ("conditions" if runway >= 9 else "decline unless new equity")

ranked = sorted(rows, key=lambda r: (-r["runway_check"], r["company"]))
for i, r in enumerate(ranked, 1):
    print(f"{i:>2}. {r['company']:<22} {r['runway_check']:>5.1f} months  {band(r['runway_check'])}")

## Exceptions the ranking does not show

- Tessaly Credit: ARR restated to $11.6M after removing one-time pilot revenue; growth restated to 40%.
- Ynderra Lending: largest customer is 46% of ARR, above the 25% concentration guideline.
- Marrowfield Credit: figures come from the v2 statement; the v1 draft is superseded.
- Restricted documents (board minutes, litigation memo) were unavailable and are not reflected in runway.

In [ ]:
flags = {r["company"]: r["topRisk"]["text"] for r in rows if any(k in r["topRisk"]["text"].lower() for k in ("restat", "concentration"))}
flags

## Sensitivity: monthly burn +20% and -20%

Runway is recomputed at base burn, at burn x 1.2 and at burn x 0.8 (cash / burn variants) and must match `analysis/sensitivity.json`.

- Under +20% burn, Halvorine Funding, Corvane Capital, Solvane Trade Credit fall below the committee's 12-month runway threshold for an unconditional proceed.
- Marrowfield Credit and Tessaly Credit are below 9 months at base burn; even a 20% burn cut only lifts them into the 9-12 month band, so they still need new equity or a liquidity covenant.

In [ ]:
sens = {r["company"]: r for r in json.loads(Path("sensitivity.json").read_text())["rows"]}
stress = []
for r in rows:
    cash, burn = r["cashUsdM"]["value"], r["monthlyBurnUsdM"]["value"]
    case = {"base": cash / burn, "burnUp20": cash / (burn * 1.2), "burnDown20": cash / (burn * 0.8)}
    for k, v in case.items():
        assert abs(sens[r["company"]]["runwayMonths"][k] - v) <= 0.1, (r["company"], k)
    stress.append((r["company"], round(case["base"], 1), round(case["burnUp20"], 1), round(case["burnDown20"], 1), band(case["burnUp20"])))
for s in sorted(stress, key=lambda s: -s[2]):
    print(f"{s[0]:<22} base {s[1]:>5.1f}  +20% {s[2]:>5.1f}  -20% {s[3]:>5.1f}  {s[4]}")